# 171Yb 65 3S1 Rydberg lifetime with rydcalc

This notebook evaluates the lifetime of
$|r\rangle = |65\,{}^3S_1, F=3/2, m_F=-3/2\rangle$ using the local `rydcalc` submodule.  The decay-rate call follows the `rydcalc/tutorial.ipynb` convention, where an environment is built with `rydcalc.environment(T_K=...)` and the lifetime is `tau = 1 / gamma`.

For this Yb171 MQDT line, asking `total_decay` to start its bound-state search at zero effective quantum number can become numerically unstable.  The calculation below therefore constructs the dipole-allowed P-state decay basis explicitly and uses `nu_min = 3.0`, `nu_max = nu_initial + 10`.  This cutoff is printed with the result.

In [1]:
from __future__ import annotations

import os
import sys
import warnings
from pathlib import Path

import numpy as np
from scipy import constants as cs

root = Path.cwd()
if not (root / "src").exists() and (root.parent / "src").exists():
    root = root.parent
if str(root / "src") not in sys.path:
    sys.path.insert(0, str(root / "src"))

mpl_cache = Path("/tmp/matplotlib-rydcalc")
mpl_cache.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("MPLCONFIGDIR", str(mpl_cache))

from neutral_yb.external.rydcalc_adapter import (  # noqa: E402
    build_yb171_atom,
    has_arc_c_extension,
    load_rydcalc,
)

rydcalc = load_rydcalc(require_c_extension=True)
from rydcalc.single_basis import state_mqdt  # noqa: E402

yb = build_yb171_atom(cpp_numerov=True, use_db=False, require_c_extension=True)
initial_state = yb.get_state((65, 0, 1.5, -1.5), tt="vlfm")

print(f"project root: {root}")
print(f"rydcalc ARC C extension available: {has_arc_c_extension()}")
print(f"state: {initial_state}")
print(f"v_exact/nub: {initial_state.nub:.8f}")
print(f"energy rel. upper HF limit: {initial_state.get_energy_Hz() / 1e12:.12f} THz")

project root: /home/eris/projects/Noise-Tolerant-Quantum-Control-Optimization
rydcalc ARC C extension available: True
state: |171Yb:64.56,L=0,F=1.5,-1.5>
v_exact/nub: 64.56106363
energy rel. upper HF limit: -0.786121460119 THz


In [2]:
def build_dipole_decay_basis(atom, state, *, nu_min: float = 3.0, nu_extra: float = 10.0):
    """Build Yb171 P-state decay channels for an S-state lifetime calculation."""
    channels = []
    nu_max = state.nub + nu_extra
    target_l = 1
    f_values = np.arange(max(0.5, state.f - 1), state.f + 1.1, 1.0)

    for f_value in f_values:
        matches = [
            item
            for item in atom.mqdt_models
            if item["L"] == target_l and item["F"] == f_value
        ]
        if not matches:
            continue

        solver = matches[0]["model"]
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", RuntimeWarning)
            nua_list, nub_list = solver.boundstatesinrange([nu_min, nu_max])

        for nua, nub in zip(nua_list, nub_list):
            energy_hz = (
                -solver.RydConst_invcm / nub**2
                + solver.ionizationlimits_invcm[solver.nulimb[0]]
            ) * 100 * cs.c
            with warnings.catch_warnings():
                warnings.simplefilter("ignore", RuntimeWarning)
                coeffs_i, coeffs_alpha = solver.channelcontributions(nub)

            if not np.all(np.isfinite(np.asarray(coeffs_i, dtype=float))):
                continue

            nuapprox = round(nub * 100) / 100
            for m_value in np.arange(state.m - 1, state.m + 1.1, 1.0):
                if not (-f_value <= m_value <= f_value):
                    continue

                final_state = state_mqdt(
                    atom,
                    (nuapprox, -1, f_value, m_value),
                    coeffs_i,
                    coeffs_alpha,
                    solver.channels,
                    energy_Hz=energy_hz,
                    tt="npfm",
                )
                final_state.pretty_str = (
                    f"|{atom.name}:{nuapprox:.2f},L=1,F={f_value:.1f},{m_value:.1f}>"
                )
                final_state.short_str = f"|{nuapprox:.2f},1,{f_value:.1f},{m_value:.1f}>"
                final_state.nua = nua
                final_state.nub = nub
                final_state.whittaker_wfct = False
                channels.append(final_state)

    return channels


def lifetime_summary(atom, state, final_states, *, temperature_k: float):
    env = rydcalc.environment(T_K=temperature_k)
    rows = []
    gamma_total = 0.0

    for final_state in final_states:
        gamma = atom.partial_decay(state, final_state, env)
        if not np.isfinite(gamma):
            continue
        gamma_total += gamma
        if gamma != 0:
            rows.append(
                {
                    "gamma_s^-1": gamma,
                    "tau_us_contribution": 1e6 / gamma,
                    "delta_GHz": (state.get_energy_Hz() - final_state.get_energy_Hz()) / 1e9,
                    "final_state": final_state.pretty_str,
                }
            )

    rows.sort(key=lambda row: row["gamma_s^-1"], reverse=True)
    return {
        "temperature_K": temperature_k,
        "gamma_s^-1": gamma_total,
        "tau_s": 1 / gamma_total,
        "tau_us": 1e6 / gamma_total,
        "nonzero_channels": len(rows),
        "top_channels": rows[:8],
    }

In [3]:
NU_MIN = 3.0
NU_EXTRA = 10.0

final_states = build_dipole_decay_basis(
    yb,
    initial_state,
    nu_min=NU_MIN,
    nu_extra=NU_EXTRA,
)
summaries = [
    lifetime_summary(yb, initial_state, final_states, temperature_k=temperature)
    for temperature in (0, 300)
]

print(f"basis cutoff: nu_min={NU_MIN}, nu_max={initial_state.nub + NU_EXTRA:.8f}")
print(f"candidate final states: {len(final_states)}")
for summary in summaries:
    print()
    print(f"T = {summary['temperature_K']:g} K")
    print(f"  gamma = {summary['gamma_s^-1']:.6e} s^-1")
    print(f"  tau   = {summary['tau_s']:.6e} s = {summary['tau_us']:.3f} us")
    print(f"  nonzero channels = {summary['nonzero_channels']}")

basis cutoff: nu_min=3.0, nu_max=74.56106363
candidate final states: 1680

T = 0 K
  gamma = 1.554823e+03 s^-1
  tau   = 6.431601e-04 s = 643.160 us
  nonzero channels = 1444

T = 300 K
  gamma = 5.596472e+03 s^-1
  tau   = 1.786840e-04 s = 178.684 us
  nonzero channels = 1680


In [4]:
for summary in summaries:
    print(f"T = {summary['temperature_K']:g} K top channels")
    for index, row in enumerate(summary["top_channels"], start=1):
        print(
            f"  {index}. gamma={row['gamma_s^-1']:.6e} s^-1, "
            f"delta={row['delta_GHz']:.3f} GHz, final={row['final_state']}"
        )
    print()

T = 0 K top channels
  1. gamma=1.958988e+02 s^-1, delta=210042.170 GHz, final=|171Yb:3.95,L=1,F=2.5,-2.5>
  2. gamma=1.813806e+02 s^-1, delta=416305.051 GHz, final=|171Yb:2.81,L=1,F=1.5,-1.5>
  3. gamma=1.209204e+02 s^-1, delta=416305.051 GHz, final=|171Yb:2.81,L=1,F=1.5,-0.5>
  4. gamma=7.835954e+01 s^-1, delta=210042.170 GHz, final=|171Yb:3.95,L=1,F=2.5,-1.5>
  5. gamma=6.306797e+01 s^-1, delta=356236.091 GHz, final=|171Yb:3.04,L=1,F=0.5,-0.5>
  6. gamma=5.757277e+01 s^-1, delta=211597.669 GHz, final=|171Yb:3.94,L=1,F=1.5,-1.5>
  7. gamma=5.345073e+01 s^-1, delta=127089.304 GHz, final=|171Yb:5.07,L=1,F=2.5,-2.5>
  8. gamma=3.838184e+01 s^-1, delta=211597.669 GHz, final=|171Yb:3.94,L=1,F=1.5,-0.5>

T = 300 K top channels
  1. gamma=5.104346e+02 s^-1, delta=-12.448 GHz, final=|171Yb:65.08,L=1,F=2.5,-2.5>
  2. gamma=4.835891e+02 s^-1, delta=11.987 GHz, final=|171Yb:64.08,L=1,F=2.5,-2.5>
  3. gamma=2.568838e+02 s^-1, delta=-16.940 GHz, final=|171Yb:65.27,L=1,F=1.5,-1.5>
  4. gamma=2.182